---
title: PyTorch基础
date: 2026-05-24 16:30:00
categories:
  - 学习
  - 机器学习
tags:
  - PyTorch
  - 深度学习
  - 较难知识
sticky: 1
---

# PyTorch 基础

这份 notebook 按照菜鸟教程 PyTorch 教程的学习路线重新组织，内容覆盖：PyTorch 简介、Tensor、自动求导、神经网络、数据加载、训练循环、线性回归、分类模型、CNN/RNN 形状理解、GPU、模型保存加载和练习。

参考页面：<https://www.runoob.com/pytorch/pytorch-tutorial.html>

学习建议：每一节先读 markdown，再运行代码。代码里写了比较细的中文注释，重点不是背 API，而是看懂每一步为什么这样写。

## 1. 环境检查

先确认 PyTorch 是否安装成功，以及当前是否能使用 CUDA GPU。后面所有代码都会使用统一的 `device`，这样有 GPU 时自动用 GPU，没有 GPU 时也能在 CPU 上跑。

In [ ]:
# 导入 PyTorch 主库，Tensor、随机数、保存加载、CUDA 检查等都在 torch 里
import torch

# torch.nn 是神经网络模块，里面有 Linear、ReLU、Conv2d、损失函数等
import torch.nn as nn

# torch.optim 里面放优化器，例如 SGD、Adam、AdamW
import torch.optim as optim

# Dataset 和 DataLoader 用于组织数据和按 batch 读取数据
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

# NumPy 常用于和 Tensor 互相转换，也常用于传统数据处理
import numpy as np

# 固定随机种子，让随机生成的数据尽量可复现
# 这样你多次运行 notebook，结果不会差得太离谱
torch.manual_seed(42)

# 打印 PyTorch 版本，排查环境问题时很有用
print("PyTorch version:", torch.__version__)

# 检查当前环境是否支持 CUDA GPU
print("CUDA available:", torch.cuda.is_available())

# 如果有 CUDA GPU，就使用 cuda；否则使用 cpu
# 后面模型和数据都会统一移动到这个 device 上
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 如果有 GPU，打印显卡名称，确认 PyTorch 能识别到它
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

### 1.1 CUDA 和 GPU：什么时候用显卡

`torch.cuda` 用来检查 GPU、统计设备数量、查看显存等。写代码时最好先判断 CUDA 是否可用。

In [ ]:
# 判断当前环境是否有可用 CUDA GPU
use_cuda = torch.cuda.is_available()

# 根据环境选择设备
device = torch.device("cuda" if use_cuda else "cpu")

# 创建 Tensor 后移动到指定设备
x = torch.randn(2, 3).to(device)

print("是否可用 CUDA：", use_cuda)
print("当前设备：", device)
print("Tensor 所在设备：", x.device)

# 如果有 GPU，可以查看 GPU 数量
if use_cuda:
    print("GPU 数量：", torch.cuda.device_count())


### 1.2 随机种子：让实验结果更稳定

训练神经网络时经常需要固定随机种子，让实验结果更容易复现。

In [ ]:
# 固定随机种子
torch.manual_seed(42)

# 两次运行这个单元时，随机数会保持一致
x1 = torch.randn(3)

# 再次设置同一个种子
torch.manual_seed(42)

# 得到和 x1 一样的随机数
x2 = torch.randn(3)

print(x1)
print(x2)
print(torch.equal(x1, x2))


## 2. PyTorch 核心概念总览

PyTorch 可以先抓住几个关键词：

- `torch`：核心张量计算，类似 NumPy，但支持 GPU 和自动求导
- `Tensor`：PyTorch 的核心数据结构，模型输入、参数、输出都用它表示
- `autograd`：自动求导系统，负责反向传播需要的梯度
- `nn.Module`：构建神经网络模型的基类
- `optim`：优化器模块，用梯度更新参数
- `Dataset/DataLoader`：数据集和数据加载器，用来按 batch 喂数据

下面从 Tensor 开始。

### 2.1 先把专业术语说清楚

很多 PyTorch 教程难懂，不是因为代码特别难，而是因为里面有很多专业术语。下面先把常见词解释清楚，后面再看代码会顺很多。

| 术语 | 人话解释 | 举例 |
| --- | --- | --- |
| 数据 | 拿来给模型学习的材料 | 图片、文字、表格、声音 |
| 样本 | 一条数据 | 一张猫的图片、一个学生的信息 |
| 特征 | 样本里用来判断的输入信息 | 房子的面积、楼层、位置 |
| 标签 | 这条样本的正确答案 | 图片是猫、房价是 100 万、成绩是及格 |
| 模型 | 一个会根据输入给出预测的函数 | 输入图片，输出“猫/狗”概率 |
| 参数 | 模型内部可以被训练改变的数字 | 神经网络里的权重和偏置 |
| 预测值 | 模型算出来的答案 | 模型预测房价是 98 万 |
| 真实值 | 数据里原本给出的正确答案 | 实际房价是 100 万 |
| 损失 | 预测值和真实值差得有多远 | 预测越离谱，损失越大 |
| 优化器 | 负责调整模型参数的工具 | Adam、SGD |
| 梯度 | 告诉参数该往哪个方向改的信号 | 类似“往左调一点会更准” |
| 训练 | 让模型反复看数据、改参数 | 多做题，多改错 |
| 推理 | 用训练好的模型做预测 | 给新图片判断是不是猫 |
| batch | 一小批样本 | 一次给模型看 32 张图片 |
| epoch | 把所有训练数据完整学一遍 | 1000 张图全看完一次就是 1 个 epoch |
| 过拟合 | 模型把训练数据背熟了，但新数据表现差 | 只会做原题，不会做新题 |
| 欠拟合 | 模型还没学明白，训练数据也做不好 | 基础概念都没掌握 |


### 什么是标签 label？

**标签就是正确答案。**

例如你要训练一个模型识别猫和狗：

```text
0 表示猫
1 表示狗
```

一张猫的图片，它的标签就是 `0`；一张狗的图片，它的标签就是 `1`。

In [ ]:
# 假设有 4 张图片，这里不真的放图片，只用随机数模拟图片特征
features = torch.randn(4, 3)

# labels 就是标签，也就是每个样本的正确答案
# 这里约定：0 表示猫，1 表示狗
labels = torch.tensor([0, 1, 0, 1])

print("特征 shape：", features.shape)
print("标签：", labels)


### 什么是特征 feature？

**特征就是模型用来判断的输入信息。**

比如预测房价时，模型不能凭空猜，它需要一些输入：面积、楼层、城市、房龄、是否靠近地铁。

这些输入信息就叫特征。在深度学习里，图片也可以看成特征：图片里的每个像素值，都是模型的输入信息。

In [ ]:
# 一个房价预测的小例子
# 每一行是一个房子样本
# 三列分别表示：面积、楼层、房龄
house_features = torch.tensor([
    [80.0, 6.0, 10.0],
    [120.0, 18.0, 3.0],
    [60.0, 2.0, 20.0],
])

# 标签是每个房子的真实价格，单位可以理解成“万元”
house_labels = torch.tensor([200.0, 420.0, 150.0])

print(house_features)
print(house_labels)


### 什么是模型 model？

**模型就是一个函数：输入一些东西，输出一个预测结果。**

可以先把模型想成这样：

```text
输入特征 -> 模型 -> 预测答案
```

例如：房子面积、楼层、房龄进入模型，模型输出预测房价。

In [ ]:
# 一个最简单的模型：输入 3 个特征，输出 1 个预测值
model = torch.nn.Linear(3, 1)

# 输入一个房子的 3 个特征：面积、楼层、房龄
one_house = torch.tensor([[80.0, 6.0, 10.0]])

# 模型给出预测结果
pred_price = model(one_house)

print("预测房价：", pred_price)


### 什么是损失 loss？

**损失就是模型错得有多离谱。**

预测值和真实标签越接近，损失越小；差距越大，损失越大。训练模型的目标就是让损失越来越小。

In [ ]:
# 假设模型预测了 3 个房价
pred = torch.tensor([98.0, 210.0, 305.0])

# 真实房价标签
true = torch.tensor([100.0, 200.0, 300.0])

# mse_loss 是均方误差，常用于回归任务
loss = torch.nn.functional.mse_loss(pred, true)

print("损失：", loss.item())


### 什么是 batch 和 epoch？

如果数据很多，模型通常不会一次性看完所有数据，而是分批看。

- **batch**：一次喂给模型的一小批数据
- **batch size**：这一小批里有多少条样本
- **epoch**：所有训练数据被完整看完一遍

比如有 1000 张图片，batch size 是 100，那么 1 个 epoch 就包含 10 个 batch。

In [ ]:
# 构造 10 条样本，每条样本有 2 个特征
x = torch.randn(10, 2)

# 构造 10 个标签
y = torch.randint(0, 2, (10,))

# TensorDataset 把特征和标签打包成数据集
dataset = torch.utils.data.TensorDataset(x, y)

# batch_size=4 表示每次取 4 条样本
loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=False)

# 遍历 DataLoader，每次拿到一个 batch
for batch_index, (batch_x, batch_y) in enumerate(loader):
    print("第几个 batch：", batch_index)
    print("batch 特征 shape：", batch_x.shape)
    print("batch 标签：", batch_y)


### 什么是梯度 gradient？

**梯度可以理解成“参数应该怎么改，损失才能变小”的方向提示。**

训练神经网络时，模型会先做预测，再计算损失，然后反向传播计算梯度，最后优化器根据梯度修改参数。

不用一开始就把梯度的数学公式弄懂。先记住：梯度是模型改正错误时需要的方向信息。

In [ ]:
# 创建一个可训练的参数 w
w = torch.tensor(2.0, requires_grad=True)

# 假设目标是让 w 接近 5
loss = (w - 5) ** 2

# backward 会计算梯度
loss.backward()

# w.grad 就是 w 的梯度
print("当前 w：", w.item())
print("当前损失：", loss.item())
print("w 的梯度：", w.grad.item())


### 什么是训练和推理？

**训练**：模型看带标签的数据，反复修改参数，让预测越来越准。

**推理**：模型已经训练好，拿它去预测新数据。

可以这样记：训练是学习阶段，需要标签，需要改参数；推理是考试阶段，不需要标签，不改参数，只输出预测。

In [ ]:
# 训练阶段：需要计算梯度，所以模型参数会被优化器更新
model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

x = torch.randn(5, 2)
y = torch.randn(5, 1)

pred = model(x)
loss = torch.nn.functional.mse_loss(pred, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()

# 推理阶段：不需要计算梯度
with torch.no_grad():
    new_x = torch.randn(1, 2)
    result = model(new_x)
    print("推理结果：", result)


### 分类、回归、标签的区别

常见任务可以先分成两类：

| 任务 | 预测什么 | 标签长什么样 | 例子 |
| --- | --- | --- | --- |
| 分类 | 预测类别 | 通常是类别编号 | 猫/狗/鸟，垃圾邮件/正常邮件 |
| 回归 | 预测连续数字 | 通常是小数或整数 | 房价、温度、销量 |

分类任务里，标签常常是 `0、1、2` 这种类别编号。回归任务里，标签常常是 `100.0、36.5、5200.0` 这种真实数值。

## 3. Tensor 张量基础

Tensor 可以理解成“支持 GPU 和自动求导的多维数组”。

常见维度：

| 数据 | 维度 | shape 示例 |
|---|---:|---|
| 标量 | 0 维 | `torch.Size([])` |
| 向量 | 1 维 | `torch.Size([5])` |
| 矩阵 | 2 维 | `torch.Size([3, 4])` |
| 单张彩色图片 | 3 维 | `torch.Size([3, 224, 224])` |
| 一批图片 | 4 维 | `torch.Size([32, 3, 224, 224])` |

In [ ]:
# 0 维 Tensor：标量，也就是单个数字
scalar = torch.tensor(3.14)

# 1 维 Tensor：向量，可以表示一个样本的多个特征
vector = torch.tensor([1, 2, 3, 4, 5])

# 2 维 Tensor：矩阵，可以表示表格数据或一批样本的特征
# arange(12) 生成 0 到 11，然后 reshape 成 3 行 4 列
matrix = torch.arange(12).reshape(3, 4)

# 3 维 Tensor：这里模拟一张图片，格式为 [channels, height, width]
image = torch.randn(3, 224, 224)

# 4 维 Tensor：这里模拟一批图片，格式为 [batch, channels, height, width]
batch_images = torch.randn(32, 3, 224, 224)

# 逐个打印 Tensor 的形状、维度数量和元素总数
for name, tensor in [
    ("scalar", scalar),
    ("vector", vector),
    ("matrix", matrix),
    ("image", image),
    ("batch_images", batch_images),
]:
    print(f"{name:12s} shape={tuple(tensor.shape)}, ndim={tensor.ndim}, numel={tensor.numel()}")

### 3.1 创建 Tensor

常用创建函数包括：`torch.tensor`、`zeros`、`ones`、`randn`、`arange`、`linspace`、`eye`。这些函数在写测试代码、初始化数据、理解 shape 时非常常用。

In [ ]:
# 从 Python 列表直接创建 Tensor
# dtype=torch.float32 表示元素使用 32 位浮点数，神经网络里最常见
a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

# 创建 2 行 3 列的全 0 Tensor
zeros = torch.zeros(2, 3)

# 创建 2 行 3 列的全 1 Tensor
ones = torch.ones(2, 3)

# 创建 2 行 3 列的随机 Tensor，元素来自标准正态分布
random_normal = torch.randn(2, 3)

# 创建等差整数序列：0, 2, 4, 6, 8
seq = torch.arange(0, 10, 2)

# 在 0 到 1 之间均匀取 5 个点
linear_space = torch.linspace(0, 1, steps=5)

# 创建 4 行 4 列单位矩阵，主对角线为 1，其余为 0
identity = torch.eye(4)

print("a =\n", a)
print("zeros =\n", zeros)
print("ones =\n", ones)
print("random_normal =\n", random_normal)
print("seq =", seq)
print("linear_space =", linear_space)
print("identity =\n", identity)

### 3.2 Tensor 的属性：shape、dtype、device

写 PyTorch 代码时，一旦报错，第一反应就是检查 `shape`、`dtype` 和 `device`。

In [ ]:
# 创建一个 4 行 5 列的随机 Tensor
x = torch.randn(4, 5)

# shape 表示形状，这里是 [4, 5]
print("shape:", x.shape)

# dtype 表示数据类型，随机浮点 Tensor 默认通常是 float32
print("dtype:", x.dtype)

# device 表示 Tensor 当前所在设备，默认在 CPU 上
print("device:", x.device)

# ndim 表示维度数量，[4, 5] 是二维
print("ndim:", x.ndim)

# numel 表示元素总数，4 * 5 = 20
print("numel:", x.numel())

### 3.3 索引、切片、布尔筛选

Tensor 的索引规则和 Python 列表、NumPy 类似，都是从 0 开始计数。

In [ ]:
# 创建一个 3 行 4 列的矩阵
x = torch.arange(12).reshape(3, 4)
print("x =\n", x)

# 取第 0 行
print("第 0 行:", x[0])

# 取第 1 行第 2 列的元素
print("第 1 行第 2 列:", x[1, 2])

# 取所有行的第 0 列
print("第 0 列:", x[:, 0])

# 取第 1 行到最后一行，并且取前 2 列
print("第 1 行到最后，前 2 列:\n", x[1:, :2])

# 布尔筛选：先构造一个包含正数和负数的向量
values = torch.tensor([1, -2, 3, -4, 5])

# mask 中正数位置为 True，其他位置为 False
mask = values > 0
print("mask:", mask)

# 用 mask 筛选出所有正数
print("positive values:", values[mask])

### 3.4 形状变换

神经网络对输入形状很敏感。常见操作有 `reshape`、`unsqueeze`、`squeeze`、`permute`、`flatten`。

In [ ]:
# 构造一个 shape 为 [2, 3, 4] 的 Tensor
# 总元素数量是 2 * 3 * 4 = 24
x = torch.arange(24).reshape(2, 3, 4)
print("原始 shape:", x.shape)

# reshape(2, 12)：把后两个维度合并
print("reshape(2, 12):", x.reshape(2, 12).shape)

# flatten()：把所有维度拉平成一维
print("flatten():", x.flatten().shape)

# flatten(start_dim=1)：保留第 0 维，从第 1 维开始拉平
print("flatten(start_dim=1):", x.flatten(start_dim=1).shape)

# unsqueeze 增加一个长度为 1 的维度
v = torch.tensor([1, 2, 3])
print("v shape:", v.shape)
print("v.unsqueeze(0):", v.unsqueeze(0).shape)
print("v.unsqueeze(1):", v.unsqueeze(1).shape)

# squeeze 删除长度为 1 的维度
y = torch.randn(1, 3, 1, 4)
print("y shape:", y.shape)
print("y.squeeze():", y.squeeze().shape)

# 图片从 HWC 转为 CHW
# 很多图片库读出来是 [height, width, channels]
img_hwc = torch.randn(224, 224, 3)

# PyTorch 卷积层一般需要 [channels, height, width]
img_chw = img_hwc.permute(2, 0, 1)
print("HWC -> CHW:", img_hwc.shape, "->", img_chw.shape)

### 3.5 Tensor 运算、广播和矩阵乘法

Tensor 支持逐元素运算、统计运算、广播和矩阵乘法。广播规则可以简单记成：从最后一个维度开始对齐；两个维度相等，或其中一个为 1，就可以广播。

In [ ]:
# 逐元素运算：相同位置的元素互相运算
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])
print("x + y:", x + y)
print("x * y:", x * y)
print("x ** 2:", x ** 2)

# 统计运算：sum、mean、max、min 很常用
m = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("m =\n", m)
print("所有元素求和:", m.sum())
print("所有元素平均值:", m.mean())
print("每一列求和 dim=0:", m.sum(dim=0))
print("每一行求和 dim=1:", m.sum(dim=1))

# 广播示例：[4, 3] + [3]
base = torch.ones(4, 3)
bias = torch.tensor([10, 20, 30])
print("广播结果:\n", base + bias)

# 矩阵乘法要求前一个矩阵的列数等于后一个矩阵的行数
# [2, 3] @ [3, 4] -> [2, 4]
a = torch.randn(2, 3)
b = torch.randn(3, 4)
c = a @ b
print("matrix multiply:", a.shape, "@", b.shape, "=", c.shape)

### 3.6 Tensor 和 NumPy、设备转换

PyTorch 经常和 NumPy 配合使用；GPU 训练时还需要在 CPU 和 CUDA 之间移动 Tensor。

In [ ]:
# NumPy 转 Tensor
numpy_array = np.array([[1, 2], [3, 4]])
tensor_from_numpy = torch.from_numpy(numpy_array)
print("from numpy:\n", tensor_from_numpy)

# Tensor 转 NumPy
# 如果 Tensor 需要梯度或在 GPU 上，推荐写成 detach().cpu().numpy()
tensor_for_numpy = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
array_from_tensor = tensor_for_numpy.detach().cpu().numpy()
print("to numpy:", array_from_tensor)

# 在指定设备上创建 Tensor
x = torch.randn(2, 3, device=device)
print("x device:", x.device)

# 如果后续需要回到 CPU，可以用 .cpu()
x_cpu = x.cpu()
print("x_cpu device:", x_cpu.device)

### 3.7 torch 顶层函数：创建、拼接、保存

`torch` 顶层模块里放着 Tensor 创建、拼接、数学运算、随机数、保存加载等基础能力。

In [ ]:
# 导入 PyTorch 主库
import torch

# 创建 Tensor：从 Python 列表变成 PyTorch Tensor
a = torch.tensor([1, 2, 3])

# 创建全 0 Tensor，常用来初始化占位数据
zeros = torch.zeros(2, 3)

# 创建标准正态分布随机 Tensor，常用于测试模型输入
random_x = torch.randn(4, 5)

# 按维度拼接 Tensor，dim=0 表示按行拼接
cat_x = torch.cat([zeros, zeros], dim=0)

# 保存 Tensor 或模型参数到文件
torch.save(a, "tensor_demo.pt")

# 读取刚才保存的 Tensor
loaded_a = torch.load("tensor_demo.pt")

print(a)
print(zeros.shape)
print(random_x.shape)
print(cat_x.shape)
print(loaded_a)


### 3.8 Tensor 自带方法：reshape、mean、sum、to

很多操作既可以写成 `torch.mean(x)`，也可以写成 `x.mean()`。后者就是 Tensor 对象自己的方法。

In [ ]:
# 创建一个 2 行 3 列的 Tensor
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

# 查看形状
print(x.shape)

# reshape 改变形状，不改变数据总数量
print(x.reshape(3, 2))

# mean 计算平均值
print(x.mean())

# sum 按列求和，dim=0 表示压缩第 0 个维度
print(x.sum(dim=0))

# to 可以移动设备或转换类型
print(x.to(dtype=torch.int64))


### 3.9 torch.linalg：矩阵和线性代数

`torch.linalg` 适合处理矩阵范数、逆矩阵、特征值、奇异值分解等线性代数问题。

In [ ]:
# 创建一个 2x2 矩阵
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

# 计算矩阵范数
print(torch.linalg.norm(A))

# 计算逆矩阵
print(torch.linalg.inv(A))

# 计算特征值
print(torch.linalg.eigvals(A))


### 3.10 torch.fft：频域变换的基本认识

`torch.fft` 主要用于信号处理、图像处理和频域分析。初学深度学习可以先知道它的用途。

In [ ]:
# 创建一个简单的一维信号
signal = torch.tensor([1.0, 0.0, -1.0, 0.0])

# 快速傅里叶变换，把时域信号变到频域
freq = torch.fft.fft(signal)

# 逆傅里叶变换，把频域结果变回时域
back = torch.fft.ifft(freq)

print(freq)
print(back)


### 3.11 torch.distributions：概率分布入门

`torch.distributions` 可以创建正态分布、分类分布等，常用于概率模型、强化学习和生成模型。

In [ ]:
# 创建一个均值为 0、标准差为 1 的正态分布
normal = torch.distributions.Normal(loc=0.0, scale=1.0)

# 从分布中采样 5 个数
samples = normal.sample((5,))

# 计算样本对应的对数概率
log_prob = normal.log_prob(samples)

print(samples)
print(log_prob)


## 4. 自动求导 Autograd

自动求导是 PyTorch 训练神经网络的关键。只要 Tensor 设置了 `requires_grad=True`，PyTorch 就会记录它参与过的运算。对最终损失调用 `backward()` 后，相关变量的 `.grad` 中会保存梯度。

In [ ]:
# 创建一个需要求梯度的标量 x
# requires_grad=True 表示 PyTorch 要追踪它的计算过程
x = torch.tensor(2.0, requires_grad=True)

# 定义函数 y = 3x^2 + 2x + 1
y = 3 * x ** 2 + 2 * x + 1

# 对 y 进行反向传播，计算 dy/dx
# y 是标量，所以可以直接 backward()
y.backward()

# 数学上 dy/dx = 6x + 2，当 x=2 时结果是 14
print("y:", y.item())
print("dy/dx:", x.grad.item())

# 注意：梯度默认会累积
# 如果再次 backward 而不清空 grad，新的梯度会加到旧梯度上
y2 = 3 * x ** 2 + 2 * x + 1
y2.backward()
print("再次 backward 后累积的 grad:", x.grad.item())

# 手动清空梯度，训练模型时通常由 optimizer.zero_grad() 完成
x.grad.zero_()
print("清空后的 grad:", x.grad.item())

### 4.1 torch.autograd：更细地控制梯度

平时常用 `loss.backward()` 就够了。需要更细控制梯度时，可以了解 `torch.autograd.grad`。

In [ ]:
# 创建需要求梯度的 Tensor
x = torch.tensor(3.0, requires_grad=True)

# 定义一个简单函数 y = x^2 + 2x
y = x ** 2 + 2 * x

# 计算 y 对 x 的梯度
grad_x = torch.autograd.grad(y, x)

print(grad_x)

# no_grad 表示这段代码不记录梯度，常用于验证和推理
with torch.no_grad():
    z = x * 10
    print(z)


## 5. 神经网络基础 nn.Module

PyTorch 中的模型通常继承 `nn.Module`。写模型时重点是两个位置：

- `__init__`：定义层
- `forward`：定义数据怎么流过这些层

In [ ]:
# 定义一个很小的多层感知机 MLP
class SimpleNet(nn.Module):
    def __init__(self):
        # 调用父类初始化，nn.Module 子类必须写
        super().__init__()

        # nn.Sequential 可以把多个层按顺序组合起来
        self.net = nn.Sequential(
            # 输入 2 个特征，输出 16 个隐藏特征
            nn.Linear(2, 16),

            # ReLU 激活函数，让模型具备非线性表达能力
            nn.ReLU(),

            # 输出 1 个值，适合做回归或二分类 logit
            nn.Linear(16, 1)
        )

    def forward(self, x):
        # forward 定义前向传播逻辑
        return self.net(x)

# 创建模型并移动到 device
model = SimpleNet().to(device)

# 构造 5 个样本，每个样本有 2 个特征
sample_x = torch.randn(5, 2).to(device)

# 调用 model(sample_x) 会自动执行 forward
sample_out = model(sample_x)
print(model)
print("input shape:", sample_x.shape)
print("output shape:", sample_out.shape)

### 5.1 torch.nn：常用层和模型容器

`torch.nn` 主要负责“搭模型”，里面有全连接层、卷积层、循环层、归一化层、损失函数和模型容器。

In [ ]:
# nn 通常这样导入
import torch.nn as nn

# Sequential 可以把多个层按顺序串起来
model = nn.Sequential(
    nn.Linear(10, 32),  # 全连接层：输入 10 维，输出 32 维
    nn.ReLU(),          # 激活函数：增加非线性表达能力
    nn.Linear(32, 2)    # 输出层：输出 2 个类别的分数
)

# 构造一批假数据：batch_size=4，每条数据 10 个特征
x = torch.randn(4, 10)

# 前向传播，得到模型输出
y = model(x)

print(y.shape)
print(model)


### 5.2 ModuleList、Sequential 和常见网络层

`torch.nn` 是搭神经网络的工具箱。`nn.Sequential` 适合简单顺序模型；`nn.ModuleList` 适合需要自己控制 forward 逻辑的模型。

In [ ]:
# ModuleList 示例：它只负责保存层，不会自动执行
class TinyBlockNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),  # 输入 4 维，输出 8 维
            nn.ReLU(),        # 激活函数
            nn.Linear(8, 2)   # 输出 2 维
        ])

    def forward(self, x):
        # ModuleList 不会自动按顺序执行，所以要自己写循环
        for layer in self.layers:
            x = layer(x)
        return x

net = TinyBlockNet().to(device)
input_x = torch.randn(3, 4).to(device)  # 3 个样本，每个 4 个特征
output = net(input_x)
print("input shape:", input_x.shape)
print("output shape:", output.shape)

### 5.3 BatchNorm 和 Dropout：让训练更稳定

BatchNorm 常用于稳定训练，Dropout 常用于减少过拟合。它们在 `model.train()` 和 `model.eval()` 下行为不同，所以训练/验证模式切换非常重要。

In [ ]:
# BatchNorm 和 Dropout 示例
class NormDropoutNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 16),
            nn.BatchNorm1d(16),  # 对 16 维特征做批归一化
            nn.ReLU(),
            nn.Dropout(p=0.5),   # 训练时随机丢弃一半特征
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.net(x)

norm_model = NormDropoutNet().to(device)
example = torch.randn(4, 10).to(device)

norm_model.train()          # 训练模式：Dropout 生效
train_out = norm_model(example)

norm_model.eval()           # 评估模式：Dropout 关闭
with torch.no_grad():
    eval_out = norm_model(example)

print("train output shape:", train_out.shape)
print("eval output shape:", eval_out.shape)

## 6. 损失函数和优化器

训练神经网络的基本套路：前向传播得到预测，计算损失，反向传播得到梯度，优化器更新参数。

In [ ]:
# 重新创建模型，避免受前面演示影响
model = SimpleNet().to(device)

# MSELoss 是均方误差损失，常用于回归任务
loss_fn = nn.MSELoss()

# Adam 是常用优化器，lr 是学习率
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 构造一个 batch：8 个样本，每个样本 2 个特征
batch_x = torch.randn(8, 2).to(device)

# 构造目标值：每个样本对应 1 个连续标签
batch_y = torch.randn(8, 1).to(device)

# 1. 前向传播：模型根据输入给出预测
pred = model(batch_x)

# 2. 计算损失：预测和真实值的差距
loss = loss_fn(pred, batch_y)

# 3. 清空旧梯度：PyTorch 的梯度默认累积
optimizer.zero_grad()

# 4. 反向传播：计算每个参数的梯度
loss.backward()

# 5. 更新参数：优化器根据梯度调整模型参数
optimizer.step()

print("one step loss:", loss.item())

### 6.1 torch.nn.functional：函数式激活和损失

`torch.nn.functional` 常简写为 `F`。它里面是无状态函数，比如激活函数、softmax、交叉熵等。

In [ ]:
# functional 通常这样导入
import torch.nn.functional as F

# logits 表示模型还没有经过 softmax 的原始输出
logits = torch.tensor([[2.0, 0.5, 0.1]])

# softmax 把原始分数变成概率
prob = F.softmax(logits, dim=1)

# target 表示正确类别编号，这里正确类别是第 0 类
target = torch.tensor([0])

# cross_entropy 内部已经包含 log_softmax，所以 logits 不需要提前 softmax
loss = F.cross_entropy(logits, target)

print(prob)
print(loss)


### 6.2 常见损失函数怎么选

损失函数和任务类型强相关：回归用 `MSELoss`/`L1Loss`；二分类用 `BCEWithLogitsLoss`；多分类用 `CrossEntropyLoss`。重点：后两个都吃 logits，不要提前 sigmoid/softmax。

In [ ]:
# 1. 回归任务：预测连续值
reg_pred = torch.tensor([[2.5], [3.0], [4.0]])
reg_target = torch.tensor([[3.0], [2.5], [5.0]])
reg_loss = nn.MSELoss()(reg_pred, reg_target)
print("MSE 回归损失:", reg_loss.item())

# 2. 二分类任务：输出一个 logit，标签是 0/1 浮点数
binary_logits = torch.tensor([[1.2], [-0.7], [0.3]])
binary_target = torch.tensor([[1.0], [0.0], [1.0]])
binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_target)
print("BCEWithLogits 二分类损失:", binary_loss.item())

# 3. 多分类任务：输出每个类别的 logits，标签是类别下标 long
multi_logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 1.5, 0.3]])
multi_target = torch.tensor([0, 1], dtype=torch.long)
multi_loss = nn.CrossEntropyLoss()(multi_logits, multi_target)
print("CrossEntropy 多分类损失:", multi_loss.item())

### 6.3 torch.optim：优化器怎么更新参数

优化器负责根据梯度更新模型参数。训练循环里通常会出现 `zero_grad()`、`backward()`、`step()` 三步。

In [ ]:
# 创建一个简单模型
model = nn.Linear(3, 1)

# Adam 是常用优化器，lr 是学习率
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 假输入和假标签
x = torch.randn(5, 3)
y_true = torch.randn(5, 1)

# 前向传播
y_pred = model(x)

# 计算均方误差损失
loss = F.mse_loss(y_pred, y_true)

# 清空上一轮梯度
optimizer.zero_grad()

# 反向传播，计算梯度
loss.backward()

# 根据梯度更新参数
optimizer.step()

print(loss.item())


## 7. 数据处理与 DataLoader

`Dataset` 描述“第 i 条数据是什么”，`DataLoader` 负责批量读取、打乱和迭代。

In [ ]:
# 构造一个简单数据集：y = 3*x1 - 2*x2 + 1 + noise
n = 1000

# X 的 shape 是 [1000, 2]，表示 1000 个样本，每个样本 2 个特征
X = torch.randn(n, 2)

# 真实权重和偏置，用来生成标签
true_w = torch.tensor([[3.0], [-2.0]])
true_b = 1.0

# 加一点噪声，让数据更像真实数据
noise = 0.2 * torch.randn(n, 1)

# 用矩阵乘法生成标签，y 的 shape 是 [1000, 1]
y = X @ true_w + true_b + noise

# TensorDataset 可以把 Tensor 包装成数据集
full_dataset = TensorDataset(X, y)

# 按 8:2 拆分训练集和验证集
train_size = int(0.8 * n)
val_size = n - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# DataLoader 每次返回一个 batch
# 训练集通常 shuffle=True，让数据顺序随机化
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 验证集不需要打乱
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# 取一个 batch 看 shape
xb, yb = next(iter(train_loader))
print("batch x shape:", xb.shape)
print("batch y shape:", yb.shape)

### 7.1 自定义 Dataset

如果数据不是现成 Tensor，通常需要自己继承 `Dataset`，实现 `__len__` 和 `__getitem__`。

In [ ]:
# 自定义 Dataset 示例
class MyRegressionDataset(Dataset):
    def __init__(self, features, labels):
        # 保存特征和标签
        self.features = features
        self.labels = labels

    def __len__(self):
        # 返回样本总数
        return len(self.features)

    def __getitem__(self, index):
        # 根据下标 index 返回一个样本
        # DataLoader 会自动调用这个方法来组成 batch
        return self.features[index], self.labels[index]

custom_dataset = MyRegressionDataset(X, y)
print("dataset length:", len(custom_dataset))
print("first sample x shape:", custom_dataset[0][0].shape)
print("first sample y shape:", custom_dataset[0][1].shape)

### 7.2 torch.utils.data：把数据分批喂给模型

数据量变大后，一般不会手写循环取数据，而是使用 `Dataset` 和 `DataLoader`。

In [ ]:
# 导入数据相关工具
from torch.utils.data import TensorDataset, DataLoader

# 构造特征和标签
features = torch.randn(100, 4)
labels = torch.randint(0, 2, (100,))

# TensorDataset 会把特征和标签按相同下标打包
dataset = TensorDataset(features, labels)

# DataLoader 负责分批、打乱和迭代数据
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 取出一个 batch 看看形状
for batch_x, batch_y in loader:
    print(batch_x.shape)
    print(batch_y.shape)
    break


### 7.3 transforms 与 torchvision：图片数据预处理

图像任务通常需要：读取图片、缩放大小、转 Tensor、归一化、数据增强。真实项目一般用 `torchvision.transforms`。如果本机没装 torchvision，这个单元会提示安装。

In [ ]:
# torchvision transforms 示例
try:
    from torchvision import transforms
    from PIL import Image

    # Compose 把多个图片处理步骤串起来
    transform = transforms.Compose([
        transforms.Resize((64, 64)),      # 缩放图片到 64x64
        transforms.ToTensor(),            # PIL 图片 -> Tensor，shape 变成 [C,H,W]，数值到 0~1
        transforms.Normalize(             # 对 RGB 三个通道归一化
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5]
        )
    ])

    # 创建一张假的 RGB 图片，用于演示 transform 流程
    fake_image = Image.new("RGB", (128, 128), color=(120, 80, 200))
    image_tensor = transform(fake_image)
    print("transformed image shape:", image_tensor.shape)
    print("value range roughly:", image_tensor.min().item(), image_tensor.max().item())
except ModuleNotFoundError as e:
    print("当前环境缺少 torchvision 或 PIL。安装命令示例：")
    print("python -m pip install torchvision pillow")
    print("原始错误:", e)

## 8. 线性回归：完整训练循环

下面用 PyTorch 写一个完整训练流程。这个例子很重要，因为大多数深度学习训练代码都遵循这个结构。

In [ ]:
# 定义线性回归模型
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # 输入 2 个特征，输出 1 个连续值
        self.linear = nn.Linear(2, 1)

    def forward(self, x):
        # 直接调用线性层
        return self.linear(x)

# 创建模型、损失函数和优化器
reg_model = LinearRegressionModel().to(device)
reg_loss_fn = nn.MSELoss()
reg_optimizer = optim.SGD(reg_model.parameters(), lr=0.05)

def evaluate_regression(model, loader):
    # 切换到评估模式
    model.eval()

    total_loss = 0.0
    total_count = 0

    # 验证阶段不需要计算梯度
    with torch.no_grad():
        for xb, yb in loader:
            # 把数据移动到同一个设备
            xb = xb.to(device)
            yb = yb.to(device)

            # 前向传播
            pred = model(xb)

            # 计算当前 batch 的平均损失
            loss = reg_loss_fn(pred, yb)

            # 累计总损失。乘以 batch 大小是为了最后得到全数据集平均 loss
            total_loss += loss.item() * xb.size(0)
            total_count += xb.size(0)

    return total_loss / total_count

# 训练 20 个 epoch
for epoch in range(1, 21):
    # 切换到训练模式
    reg_model.train()

    total_loss = 0.0
    total_count = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        # 1. 前向传播
        pred = reg_model(xb)

        # 2. 计算损失
        loss = reg_loss_fn(pred, yb)

        # 3. 清空旧梯度
        reg_optimizer.zero_grad()

        # 4. 反向传播
        loss.backward()

        # 5. 更新模型参数
        reg_optimizer.step()

        # 统计训练损失
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

    train_loss = total_loss / total_count
    val_loss = evaluate_regression(reg_model, val_loader)

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

# 查看学到的权重和偏置，它们应该接近 [3, -2] 和 1
print("learned weight:", reg_model.linear.weight.detach().cpu())
print("learned bias:", reg_model.linear.bias.detach().cpu())

## 9. 第一个分类神经网络

多分类任务中，模型最后一层输出每个类别的 logit。使用 `CrossEntropyLoss` 时不要提前做 `softmax`，损失函数内部会处理。

In [ ]:
# 构造一个三分类二维数据集
num_samples = 600
num_classes = 3

# 三个类别中心点
centers = torch.tensor([[2.0, 2.0], [-2.0, 1.0], [0.0, -2.0]])

# 生成标签 0、1、2，每个类别数量相同
labels = torch.arange(num_classes).repeat_interleave(num_samples // num_classes)

# 每个样本 = 类别中心 + 随机噪声
features = centers[labels] + 0.6 * torch.randn(num_samples, 2)

# 包装数据集并划分训练/验证
cls_dataset = TensorDataset(features, labels)
cls_train, cls_val = random_split(cls_dataset, [480, 120])
cls_train_loader = DataLoader(cls_train, batch_size=32, shuffle=True)
cls_val_loader = DataLoader(cls_val, batch_size=64)

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        # 输入二维点，输出 3 个类别的 logits
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.net(x)

classifier = Classifier().to(device)
cls_loss_fn = nn.CrossEntropyLoss()
cls_optimizer = optim.Adam(classifier.parameters(), lr=0.03)

def accuracy(model, loader):
    # 计算准确率的辅助函数
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            # logits shape: [batch_size, num_classes]
            logits = model(xb)

            # 取每行最大值所在下标作为预测类别
            pred = logits.argmax(dim=1)

            correct += (pred == yb).sum().item()
            total += yb.numel()

    return correct / total

for epoch in range(1, 16):
    classifier.train()

    for xb, yb in cls_train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = classifier(xb)
        loss = cls_loss_fn(logits, yb)

        cls_optimizer.zero_grad()
        loss.backward()
        cls_optimizer.step()

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d} | val_acc={accuracy(classifier, cls_val_loader):.3f}")

### 9.1 自编码器 AutoEncoder：先压缩再还原

自编码器学习把输入压缩到低维表示，再重建回原输入。它常用于降维、去噪、异常检测等。

In [ ]:
# 一个非常小的自编码器
class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(10, 6), nn.ReLU(), nn.Linear(6, 3))
        self.decoder = nn.Sequential(nn.Linear(3, 6), nn.ReLU(), nn.Linear(6, 10))

    def forward(self, x):
        z = self.encoder(x)       # 压缩表示
        recon = self.decoder(z)   # 重建输入
        return recon

ae = AutoEncoder().to(device)
ae_x = torch.randn(5, 10).to(device)
ae_recon = ae(ae_x)

# 自编码器训练时，目标通常就是输入本身
recon_loss = nn.MSELoss()(ae_recon, ae_x)
print("reconstructed shape:", ae_recon.shape)
print("reconstruction loss:", recon_loss.item())

### 9.2 GAN：生成器和判别器的基本结构

GAN 有两个模型：生成器 Generator 和判别器 Discriminator。生成器试图生成假样本骗过判别器；判别器试图区分真实样本和假样本。这里不完整训练，只演示结构。

In [ ]:
# GAN 的最小结构示意
class Generator(nn.Module):
    def __init__(self, noise_dim=5, data_dim=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(noise_dim, 16), nn.ReLU(), nn.Linear(16, data_dim))

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, data_dim=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(data_dim, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        # 输出 logit，训练时配合 BCEWithLogitsLoss
        return self.net(x)

generator = Generator().to(device)
discriminator = Discriminator().to(device)

noise = torch.randn(4, 5).to(device)
fake_data = generator(noise)
fake_logits = discriminator(fake_data)

print("fake data shape:", fake_data.shape)
print("fake logits shape:", fake_logits.shape)

## 10. CNN 和 RNN 的形状理解

菜鸟教程后续会介绍 CNN、RNN、LSTM、GRU、Transformer 等模型。这里先不训练复杂模型，只用小例子理解输入输出 shape。

In [ ]:
# CNN 示例：Conv2d 接收 [batch, channels, height, width]
cnn = nn.Sequential(
    # 输入通道 3，输出通道 8，卷积核 3x3，padding=1 保持高宽不变
    nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1),
    nn.ReLU(),

    # 最大池化让高宽减半：32x32 -> 16x16
    nn.MaxPool2d(kernel_size=2),

    # 把特征图拉平，方便接全连接层
    nn.Flatten(),

    # 8 个通道，每个通道 16x16，所以输入维度是 8*16*16
    nn.Linear(8 * 16 * 16, 10)
).to(device)

# 模拟 4 张 32x32 RGB 图片
fake_images = torch.randn(4, 3, 32, 32).to(device)
cnn_out = cnn(fake_images)
print("CNN input shape:", fake_images.shape)
print("CNN output shape:", cnn_out.shape)

# RNN 示例：batch_first=True 时输入 shape 是 [batch, seq_len, input_size]
rnn = nn.RNN(input_size=5, hidden_size=8, batch_first=True).to(device)

# 模拟 4 条序列，每条序列长度为 6，每个时间步有 5 个特征
fake_sequence = torch.randn(4, 6, 5).to(device)
rnn_out, hidden = rnn(fake_sequence)
print("RNN input shape:", fake_sequence.shape)
print("RNN output shape:", rnn_out.shape)
print("RNN hidden shape:", hidden.shape)

### 10.1 迁移学习：借用训练好的模型

迁移学习的核心：使用已有模型学到的通用特征，只替换最后的分类头。真实图像项目里很常见。

In [ ]:
# 迁移学习结构示例
# 为避免下载预训练权重，这里只演示“冻结特征层 + 替换分类头”的写法
try:
    from torchvision import models

    resnet = models.resnet18(weights=None)  # 不下载权重，只创建结构

    # 冻结原模型参数：它们不参与训练
    for param in resnet.parameters():
        param.requires_grad = False

    # 替换最后的全连接层，把输出类别改成 5 类
    in_features = resnet.fc.in_features
    resnet.fc = nn.Linear(in_features, 5)

    trainable_params = [p for p in resnet.parameters() if p.requires_grad]
    print("可训练参数组数量:", len(trainable_params))
    print("新的分类头:", resnet.fc)
except ModuleNotFoundError as e:
    print("需要 torchvision 才能运行迁移学习示例。")
    print("安装命令示例：python -m pip install torchvision")
    print("原始错误:", e)

### 10.2 LSTM / GRU：处理序列数据

RNN、LSTM、GRU 用来处理序列数据，例如文本、时间序列、语音特征。先抓住 shape：如果 `batch_first=True`，输入通常是 `[batch, seq_len, input_size]`。

In [ ]:
# LSTM 和 GRU 的 shape 示例
batch_size = 4
seq_len = 6
input_size = 5
hidden_size = 8

# 4 条序列，每条长度 6，每个时间步 5 个特征
sequence = torch.randn(batch_size, seq_len, input_size).to(device)

# LSTM 返回 output，以及 (h_n, c_n)
lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True).to(device)
lstm_out, (h_n, c_n) = lstm(sequence)
print("LSTM output shape:", lstm_out.shape)
print("LSTM h_n shape:", h_n.shape)
print("LSTM c_n shape:", c_n.shape)

# GRU 返回 output 和 h_n，没有 c_n
gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True).to(device)
gru_out, gru_h = gru(sequence)
print("GRU output shape:", gru_out.shape)
print("GRU h_n shape:", gru_h.shape)

### 10.3 Embedding：把文字编号变成向量

词嵌入把“词的编号”转换成“可训练向量”。文本模型通常先做分词和编号，再通过 `nn.Embedding` 得到向量序列。

In [ ]:
# Embedding 示例：把单词编号变成向量
vocab = {"<pad>": 0, "good": 1, "bad": 2, "movie": 3}

# 假设一句话是 good movie，对应编号 [1, 3]
token_ids = torch.tensor([[vocab["good"], vocab["movie"]]], dtype=torch.long).to(device)

# num_embeddings 是词表大小，embedding_dim 是每个词向量的维度
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=4).to(device)
embedded = embedding(token_ids)

print("token ids shape:", token_ids.shape)
print("embedded shape:", embedded.shape)
print("embedded tensor:\n", embedded)

### 10.4 注意力机制和 Transformer 的最小直觉

注意力机制可以理解为：当前位置处理信息时，应该重点关注序列中的哪些位置。Transformer 是现代 NLP 和大模型的基础结构。

In [ ]:
# MultiheadAttention 最小示例
embed_dim = 8
num_heads = 2
attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True).to(device)

# 输入 shape: [batch, seq_len, embed_dim]
x_attn = torch.randn(2, 4, embed_dim).to(device)

# 自注意力中 query、key、value 都来自同一个 x
attn_out, attn_weights = attention(x_attn, x_attn, x_attn)

print("attention input shape:", x_attn.shape)
print("attention output shape:", attn_out.shape)
print("attention weights shape:", attn_weights.shape)

## 11. 学习率调度、保存加载、推理模式

训练中常见工程操作包括：调整学习率、保存模型参数、加载模型、推理时关闭梯度。

In [ ]:
# 学习率调度器示例
# StepLR 表示每隔 step_size 个 epoch，把学习率乘以 gamma
optimizer_demo = optim.SGD(reg_model.parameters(), lr=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer_demo, step_size=2, gamma=0.5)

for epoch in range(1, 6):
    # 这里只演示学习率变化，不进行真实训练
    current_lr = optimizer_demo.param_groups[0]["lr"]
    print(f"epoch {epoch}, lr before step = {current_lr}")

    # scheduler.step() 通常在每个 epoch 结束后调用
    scheduler.step()

# 保存模型参数
save_path = "pytorch_classifier_state_dict.pt"
torch.save(classifier.state_dict(), save_path)
print("model saved to:", save_path)

# 加载模型参数时，需要先创建相同结构的模型
loaded_classifier = Classifier().to(device)
loaded_classifier.load_state_dict(torch.load(save_path, map_location=device))

# 推理前切换到 eval 模式
loaded_classifier.eval()

# 推理时使用 no_grad，节省显存和计算
with torch.no_grad():
    test_points = torch.tensor([[2.0, 2.0], [-2.0, 1.0], [0.0, -2.0]], device=device)
    logits = loaded_classifier(test_points)

    # 推理阶段可以用 softmax 把 logits 转成概率
    probs = torch.softmax(logits, dim=1)
    pred = probs.argmax(dim=1)

print("probabilities:\n", probs.cpu())
print("predicted classes:", pred.cpu().tolist())

### 11.1 torch.save / torch.load：保存和加载模型

训练完模型后，通常保存 `state_dict`，也就是模型参数字典。

In [ ]:
# 定义一个小模型
model = nn.Linear(2, 1)

# 保存模型参数，不建议直接保存整个模型对象
torch.save(model.state_dict(), "linear_state_dict.pt")

# 创建同结构的新模型
new_model = nn.Linear(2, 1)

# 加载参数
new_model.load_state_dict(torch.load("linear_state_dict.pt"))

# 切换到评估模式，关闭 Dropout、固定 BatchNorm 行为
new_model.eval()

print(new_model.state_dict().keys())


### 11.2 no_grad 和 inference_mode：验证和推理时关闭梯度

验证和推理时不需要计算梯度，关闭梯度记录可以省显存、提速度。

In [ ]:
# 创建一个简单模型和输入
model = nn.Linear(4, 2)
x = torch.randn(1, 4)

# no_grad：不记录梯度，常用于验证阶段
with torch.no_grad():
    y1 = model(x)

# inference_mode：更彻底的推理模式，适合纯推理场景
with torch.inference_mode():
    y2 = model(x)

print(y1)
print(y2)


### 11.3 混合精度训练 AMP：更快更省显存

混合精度训练使用较低精度加速计算、减少显存。GPU 上常用，CPU 上不一定有收益。下面给的是标准写法骨架。

In [ ]:
# AMP 标准写法骨架
if torch.cuda.is_available():
    amp_model = SimpleNet().to(device)
    amp_optimizer = optim.Adam(amp_model.parameters(), lr=0.01)
    amp_loss_fn = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler()

    amp_x = torch.randn(8, 2).to(device)
    amp_y = torch.randn(8, 1).to(device)
    amp_optimizer.zero_grad()

    # autocast 自动选择合适精度执行部分运算
    with torch.cuda.amp.autocast():
        amp_pred = amp_model(amp_x)
        amp_loss = amp_loss_fn(amp_pred, amp_y)

    scaler.scale(amp_loss).backward()
    scaler.step(amp_optimizer)
    scaler.update()
    print("AMP loss:", amp_loss.item())
else:
    print("当前没有 CUDA，跳过 AMP 运行示例。")

### 11.4 TorchScript / ONNX：模型导出

部署时常需要导出模型。TorchScript 是 PyTorch 自己的中间表示，ONNX 是更通用的模型交换格式。这里演示 TorchScript；ONNX 需要额外安装 `onnx`。

In [ ]:
# TorchScript trace 示例
script_model = SimpleNet().to(device)
example_input = torch.randn(1, 2).to(device)

# trace 会用一个示例输入记录模型计算图
traced_model = torch.jit.trace(script_model, example_input)
traced_path = "simple_net_traced.pt"
traced_model.save(traced_path)
print("TorchScript model saved to:", traced_path)

# ONNX 导出骨架：需要安装 onnx
# torch.onnx.export(script_model, example_input, "simple_net.onnx", input_names=["input"], output_names=["output"])

### 11.5 分布式训练：多卡训练的基本概念

分布式训练用于多 GPU 或多机器训练。初学不用马上写完整代码，但要知道：`rank` 是当前进程编号，`world_size` 是总进程数，`backend` 是通信后端。完整 DDP 通常要用 `torchrun` 从命令行启动。

In [ ]:
# 分布式训练不适合在单个 notebook 单元里直接启动，这里只放概念变量
world_size = 1  # 总进程数。单机单卡时可以理解为 1
rank = 0        # 当前进程编号。只有一个进程时就是 0
backend = "gloo"  # CPU 可用 gloo，NVIDIA GPU 常用 nccl

print("world_size:", world_size)
print("rank:", rank)
print("backend:", backend)
print("完整 DDP 训练通常需要使用 torchrun 从命令行启动。")

## 12. 常见问题和调试习惯

初学 PyTorch 最常见的问题：

- shape 不匹配：多打印 `x.shape`
- dtype 不对：分类标签通常要 `torch.long`
- device 不一致：模型和数据都要放到同一个 `device`
- 忘记清梯度：训练时每个 batch 前要 `optimizer.zero_grad()`
- 训练和推理模式混用：训练用 `model.train()`，验证/推理用 `model.eval()`
- 推理忘记关闭梯度：使用 `with torch.no_grad():`

In [ ]:
# 一个小的调试辅助函数：打印 Tensor 的关键信息
def describe_tensor(name, tensor):
    # name 是变量名，tensor 是要检查的 Tensor
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, device={tensor.device}")

# 示例：检查一个 batch 的输入和标签
describe_tensor("xb", xb)
describe_tensor("yb", yb)

# 检查模型参数在哪个设备上
first_param = next(classifier.parameters())
describe_tensor("first model parameter", first_param)

### 12.1 模型评估：准确率和混淆矩阵

评估不仅看 loss，也要看准确率、混淆矩阵、召回率、F1 等。入门阶段先掌握准确率和混淆矩阵的手写版本。

In [ ]:
# 手写一个简单混淆矩阵
# confusion[i, j] 表示真实类别为 i、预测类别为 j 的数量
y_true = torch.tensor([0, 1, 2, 1, 0, 2, 2])
y_pred = torch.tensor([0, 2, 2, 1, 0, 0, 2])
num_classes = 3
confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)

for t, p in zip(y_true, y_pred):
    confusion[t, p] += 1

accuracy_value = (y_true == y_pred).float().mean().item()
print("accuracy:", accuracy_value)
print("confusion matrix:\n", confusion)

## 13. 练习题

建议先自己写，再看下面参考答案。

1. Tensor 练习：创建一个 `[3, 4]` Tensor，内容为 0 到 11，输出第 2 行、第 3 列、每一行的和。
2. Autograd 练习：令 `y = x^3 + 2x`，当 `x = 3` 时计算 `dy/dx`。
3. Dataset 练习：自己写一个 `Dataset`，返回 `(feature, label)`。
4. 训练循环练习：把“清梯度、反向传播、更新参数”三步封装进一个函数。

## 14. 练习参考答案

In [ ]:
# 练习 1：Tensor 创建、索引和统计
exercise_x = torch.arange(12).reshape(3, 4)
print("exercise_x =\n", exercise_x)

# 第 2 行对应下标 1
print("第 2 行:", exercise_x[1])

# 第 3 列对应下标 2
print("第 3 列:", exercise_x[:, 2])

# dim=1 表示按行求和
print("每一行的和:", exercise_x.sum(dim=1))

# 练习 2：自动求导
exercise_x = torch.tensor(3.0, requires_grad=True)
exercise_y = exercise_x ** 3 + 2 * exercise_x
exercise_y.backward()
print("dy/dx:", exercise_x.grad.item())
print("数学结果应该是 3*x^2 + 2 =", 3 * 3**2 + 2)

# 练习 3：自定义 Dataset
class PracticeDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

practice_dataset = PracticeDataset(X, y)
print("practice dataset length:", len(practice_dataset))

# 练习 4：封装单个 epoch 的训练逻辑
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    # 切换到训练模式
    model.train()

    total_loss = 0.0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)
        loss = loss_fn(pred, yb)

        # 三个关键步骤：清梯度、反向传播、更新参数
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

    return total_loss / total_count

# 用线性回归模型测试一下函数能否正常运行
practice_model = LinearRegressionModel().to(device)
practice_loss_fn = nn.MSELoss()
practice_optimizer = optim.SGD(practice_model.parameters(), lr=0.05)
practice_loss = train_one_epoch(practice_model, train_loader, practice_loss_fn, practice_optimizer, device)
print("practice one epoch loss:", practice_loss)

## 15. 后续学习路线

完成这份 notebook 后，可以继续按下面顺序学习：

1. 张量高级操作：拼接、堆叠、广播、einsum
2. Autograd 深入：计算图、detach、no_grad、自定义 Function
3. CNN：卷积、池化、图像分类
4. RNN / LSTM / GRU：序列建模
5. Transformer：注意力机制和编码器结构
6. torchvision：数据集、图像增强、预训练模型
7. 模型部署：TorchScript、ONNX、API 推理服务

### 15.1 进阶内容怎么继续学

我的建议顺序：

1. 做图像：先学 `torchvision transforms`、CNN、迁移学习、批归一化。
2. 做文本：先学 Embedding、LSTM/GRU、注意力机制、Transformer。
3. 提高训练效果：学损失函数、优化器、学习率调度器、模型评估与调试。
4. 工程部署：学保存加载、TorchScript、ONNX、混合精度。
5. GAN、自编码器、分布式训练可以后学，它们更偏专题和工程进阶。

这一节的代码主要是“骨架和直觉”。真正掌握的方法，还是把每个专题拿一个小数据集完整训练一遍。